# A1.8 · Backtracking: subsets, permutations, combinations, pruning, constraint checking

**ID:** A1.8 · **Target:** 60 min · **Plan week:** 5 (pattern notebook, Tue); second pass in week 13 · **Part:** A

**The drill.** This is a 60-minute study notebook. Every backtracking solution follows the same skeleton: *choose,
recurse, un-choose*, plus a rule for **what to try next** and a rule for **when to stop early** (pruning). Fill in the
three canonical enumerations (subsets, permutations, combinations), then five warm-ups (8-15 min each): subsets
with duplicates, combination sum, generating balanced parentheses, N-Queens and word search on a grid. Results are
compared as **canonical sorted lists**, so you may produce them in any order, but duplicates count as errors. The
references are `itertools`, filter-everything brute forces and OEIS counts.

**Rules.** Python standard library only. Do not use `itertools` inside your solutions; the tests use it as the
reference. Fill in every `raise NotImplementedError`.

## Sources
- Python docs, `itertools.combinations` / `permutations` / `product` (reference enumerations and their counts):
  https://docs.python.org/3/library/itertools.html
- T. H. Cormen, C. E. Leiserson, R. L. Rivest, C. Stein, *Introduction to Algorithms*, 4th ed., MIT Press, 2022
  (ISBN 9780262046305), Ch. 4 "Divide-and-Conquer" (recursion trees, for counting the nodes of a backtracking tree).
  https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/
- OEIS A000170, *Number of ways of placing n nonattacking queens on an n × n board*: https://oeis.org/A000170 ;
  OEIS A000108, *Catalan numbers* (number of balanced strings of n bracket pairs): https://oeis.org/A000108
- Tech Interview Handbook, *Recursion* cheatsheet (backtracking, memoization, recursion-depth corner cases):
  https://www.techinterviewhandbook.org/algorithms/recursion/ (yangshun/tech-interview-handbook @ e1d28e8, MIT).
- NeetCode 150 problem data (practice list): https://neetcode.io/practice ,
  https://github.com/neetcode-gh/leetcode @ d5ebb3e `.problemSiteData.json`.

In [ ]:
import copy
import itertools
import math
import random

from prep import Session, check

s = Session("A1.8", target_min=60)
random.seed(0)


def run_tests(fn, cases, name=None):
    """cases: list of (args_tuple, expected). Stops at the first failing case and prints it."""
    name = name or fn.__name__
    for i, (args, want) in enumerate(cases):
        got = fn(*copy.deepcopy(args))
        assert got == want, f"{name} case {i}: args={args!r}\n  got  {got!r}\n  want {want!r}"
    print(f"ok  {name}: {len(cases)} cases")


def stress(fn, brute, gen, n=300, seed=0, name=None):
    """Compare fn against a trusted brute force on n random small inputs; gen(rng) returns an args tuple."""
    name = name or fn.__name__
    rng = random.Random(seed)
    for t in range(n):
        args = gen(rng)
        want = brute(*copy.deepcopy(args))
        got = fn(*copy.deepcopy(args))
        assert got == want, f"{name} vs brute force, trial {t}: args={args!r}\n  got  {got!r}\n  want {want!r}"
    print(f"ok  {name} vs brute force: {n} random cases")


def canon(f, inner_sort=True):
    """Wrap an enumerator: returns the sorted list of results (each as a tuple). Duplicates are kept, so they fail."""
    def w(*args):
        return sorted(tuple(sorted(x)) if inner_sort else tuple(x) for x in f(*args))
    w.__name__ = f.__name__
    return w


def distinct_list(r, max_len=6):
    return r.sample(range(-5, 10), r.randint(0, max_len))

## 1. The skeleton: choose, recurse, un-choose
**Intuition.** Backtracking is a DFS over a **decision tree**. Each level makes one choice, a leaf is a complete
candidate, and you undo the choice when you return (`path.pop()`), so a single `path` list is shared by the whole search.
The work is (number of tree nodes) × (work per node). Pruning removes whole subtrees as early as possible.

**Signals:** "return **all** subsets/permutations/combinations/partitions/placements", "generate every valid ...",
small input ($n \le 10$-$20$), constraint satisfaction (N-Queens, Sudoku), path search on a grid with no revisits.

```python
def backtrack(state):
    if is_complete(state): record(copy of state); return
    for choice in candidates(state):        # in an order that makes pruning and de-duplication easy
        if not feasible(state, choice): continue   # (or `break` if the candidates are sorted)
        apply(choice); backtrack(state); undo(choice)
```

**Template A: all subsets of distinct values.** At each index, the next element to add is chosen from
`nums[start:]`. Every node of the tree is a subset, so record the path at every node.

In [ ]:
def subsets(nums):
    raise NotImplementedError  # YOUR CODE HERE


def subsets_brute(nums):
    return sorted(tuple(sorted(c)) for k in range(len(nums) + 1) for c in itertools.combinations(nums, k))


run_tests(canon(subsets), [(([1, 2, 3],), subsets_brute([1, 2, 3])), (([],), [()])])
check("subsets count is 2^n", len(subsets([1, 2, 3, 4, 5])) == 32)
stress(canon(subsets), subsets_brute, lambda r: (distinct_list(r),))

**Template B: all permutations of distinct values.** At each depth, try every unused element. Track the used ones
with a boolean list, which is $O(1)$ per check.

In [ ]:
def permutations(nums):
    raise NotImplementedError  # YOUR CODE HERE


perm_brute = lambda nums: sorted(itertools.permutations(nums))
run_tests(canon(permutations, inner_sort=False), [(([1, 2, 3],), perm_brute([1, 2, 3])), (([],), [()]), (([7],), [(7,)])])
stress(canon(permutations, inner_sort=False), perm_brute, lambda r: (distinct_list(r, 5),))

**Template C: all k-element combinations of `1..n`.** Like subsets, but record only at depth k, and **prune**:
when fewer elements remain than you still need, stop the loop early.

In [ ]:
def combinations(n, k):
    raise NotImplementedError  # YOUR CODE HERE


comb_brute = lambda n, k: sorted(itertools.combinations(range(1, n + 1), k))
run_tests(canon(combinations), [((4, 2), comb_brute(4, 2)), ((3, 0), [()]), ((3, 3), [(1, 2, 3)]), ((2, 3), [])])
stress(canon(combinations), comb_brute, lambda r: (lambda n: (n, r.randint(0, n + 1)))(r.randint(0, 7)))

## 2. De-duplication and pruning
### Warm-up 2.1: subsets when values repeat (~10 min)
`nums` may contain duplicates. Return every **distinct** sub-multiset exactly once. *Hint:* sort first. At a given
depth, skip `nums[i]` if it equals `nums[i-1]` and `i > start`: choosing the second copy at the same depth would
rebuild a subtree you already explored.

In [ ]:
def subsets_with_dup(nums):
    raise NotImplementedError  # YOUR CODE HERE


def subsets_dup_brute(nums):
    return sorted({tuple(sorted(c)) for k in range(len(nums) + 1) for c in itertools.combinations(nums, k)})


run_tests(canon(subsets_with_dup), [(([1, 2, 2],), [(), (1,), (1, 2), (1, 2, 2), (2,), (2, 2)]), (([0],), [(), (0,)]),
                                    (([2, 2, 2],), [(), (2,), (2, 2), (2, 2, 2)])])
stress(canon(subsets_with_dup), subsets_dup_brute, lambda r: ([r.randint(0, 3) for _ in range(r.randint(0, 7))],))

### Warm-up 2.2: combination sum with unlimited reuse (~12 min)
`cands` are **distinct positive** integers. Return every multiset of them (each value may be used any number of
times) that sums to `target`, each multiset once. *Hint:* sort, and pass `start = i` (not `i + 1`) to allow reuse.
Once `cands[i]` exceeds the remaining amount, `break`: every later candidate is bigger.

In [ ]:
def combination_sum(cands, target):
    raise NotImplementedError  # YOUR CODE HERE


def combination_sum_brute(cands, target):
    out = set()
    for k in range(target // min(cands) + 1 if cands else 1):
        for c in itertools.combinations_with_replacement(sorted(cands), k):
            if sum(c) == target:
                out.add(c)
    return sorted(out)


run_tests(canon(combination_sum), [(([2, 3, 6, 7], 7), [(2, 2, 3), (7,)]), (([2, 3, 5], 8), [(2, 2, 2, 2), (2, 3, 3), (3, 5)]),
                                   (([2], 1), []), (([1], 2), [(1, 1)])])
stress(canon(combination_sum), combination_sum_brute,
       lambda r: (r.sample(range(1, 8), r.randint(1, 4)), r.randint(1, 10)))

### Warm-up 2.3: all balanced strings of n bracket pairs (~8 min)
Return every string of `n` `(` and `n` `)` that is balanced. *Pruning rule:* add `(` while fewer than `n` are open,
and add `)` only while it would not close more than were opened. The tree then contains only valid prefixes, and
the count is the Catalan number $C_n$ (OEIS A000108).

In [ ]:
def gen_parens(n):
    raise NotImplementedError  # YOUR CODE HERE


def gen_parens_brute(n):
    def ok(s_):
        bal = 0
        for ch in s_:
            bal += 1 if ch == "(" else -1
            if bal < 0:
                return False
        return bal == 0
    return sorted("".join(p) for p in itertools.product("()", repeat=2 * n) if ok(p))


sorted_out = lambda f: (lambda *a: sorted(f(*a)))
run_tests(sorted_out(gen_parens), [((1,), ["()"]), ((2,), ["(())", "()()"]), ((0,), [""])], name="gen_parens")
stress(sorted_out(gen_parens), gen_parens_brute, lambda r: (r.randint(0, 6),), n=30, name="gen_parens")
catalan = [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862]              # OEIS A000108
check("gen_parens counts are Catalan numbers", [len(gen_parens(n)) for n in range(10)] == catalan)

## 3. Constraint checking
### Warm-up 3.1: count N-Queens placements (~12 min)
Count the ways to place `n` queens on an `n × n` board so that no two share a row, column or diagonal. Place one queen
per row and keep three sets: columns, diagonals `r - c` and anti-diagonals `r + c`. Each conflict test is then $O(1)$.

In [ ]:
def n_queens(n):
    raise NotImplementedError  # YOUR CODE HERE


def n_queens_brute(n):                        # one queen per row and column = a permutation; check diagonals only
    return sum(1 for p in itertools.permutations(range(n))
               if len({r - c for r, c in enumerate(p)}) == n and len({r + c for r, c in enumerate(p)}) == n)


oeis_a000170 = [1, 1, 0, 0, 2, 10, 4, 40, 92]                   # n = 0..8
run_tests(n_queens, [((n,), v) for n, v in enumerate(oeis_a000170)])
stress(n_queens, n_queens_brute, lambda r: (r.randint(0, 7),), n=20)

### Warm-up 3.2: find a word on a letter grid (~15 min)
`board` is a list of strings. Can `word` be spelled by a path of 4-adjacent cells that uses each cell at most once?
Mark a cell as used **before** you recurse and unmark it **after**. Prune immediately on a letter mismatch. An empty
word is always found.

In [ ]:
def word_search(board, word):
    raise NotImplementedError  # YOUR CODE HERE


def word_search_brute(board, word):           # explicit-stack search over (cell, index, visited set)
    if not word:
        return True
    h, w = len(board), len(board[0])
    stack = [((r, c), 1, frozenset([(r, c)])) for r in range(h) for c in range(w) if board[r][c] == word[0]]
    while stack:
        (r, c), i, seen = stack.pop()
        if i == len(word):
            return True
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < h and 0 <= nc < w and (nr, nc) not in seen and board[nr][nc] == word[i]:
                stack.append(((nr, nc), i + 1, seen | {(nr, nc)}))
    return False


B = ["ABCE", "SFCS", "ADEE"]
run_tests(word_search, [((B, "ABCCED"), True), ((B, "SEE"), True), ((B, "ABCB"), False),   # ABCB would reuse B
                        ((["a"], "a"), True), ((["a"], "aa"), False), ((["ab"], ""), True)])
stress(word_search, word_search_brute,
       lambda r: (["".join(r.choice("ab") for _ in range(w)) for w in [r.randint(1, 4)] for _ in range(r.randint(1, 4))],
                  "".join(r.choice("ab") for _ in range(r.randint(1, 6)))), n=500)

## Say it out loud (complexity and correctness)
1. Give the output size and the total time for subsets, permutations and k-combinations of $n$ items. Why is the
   time "output size × n"?
2. Why must you append `path[:]` (a copy), not `path`?
3. Subsets with duplicates: why is the skip condition `i > start` and not `i > 0`?
4. How many balanced strings of n pairs are there, asymptotically? How does that compare with the $4^n$ brute force?
5. N-Queens: what is a simple upper bound on the work, and what do the three sets buy you?
6. Word search: time complexity for an $R \times C$ board and a word of length $L$? Which pruning helps most in practice?
7. When does backtracking turn into DP (memoization)? Give an example from this notebook's neighbours.

<!-- practice:start -->
## Practice list (solve these on LeetCode)

NeetCode 150 problems for this topic, generated from `neetcode-gh/leetcode@d5ebb3e` `.problemSiteData.json` (https://neetcode.io/practice). **B75** = also in Blind 75. Tick them off in your session copy; the full checklist is `A_coding/A1_problem_list.md`. Grind 75 (https://www.techinterviewhandbook.org/grind75/) is an alternative ordering of a similar set.

**Backtracking** · plan week 13 · 9 problems

- [ ] [Subsets](https://leetcode.com/problems/subsets/) · Medium
- [ ] [Combination Sum](https://leetcode.com/problems/combination-sum/) · Medium · **B75**
- [ ] [Permutations](https://leetcode.com/problems/permutations/) · Medium
- [ ] [Subsets II](https://leetcode.com/problems/subsets-ii/) · Medium
- [ ] [Combination Sum II](https://leetcode.com/problems/combination-sum-ii/) · Medium
- [ ] [Word Search](https://leetcode.com/problems/word-search/) · Medium · **B75**
- [ ] [Palindrome Partitioning](https://leetcode.com/problems/palindrome-partitioning/) · Medium
- [ ] [Letter Combinations of a Phone Number](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) · Medium
- [ ] [N Queens](https://leetcode.com/problems/n-queens/) · Hard

**Extras (not in NeetCode 150; close to the warm-ups above)**

- [ ] [Combinations](https://leetcode.com/problems/combinations/) · Medium
- [ ] [N Queens II](https://leetcode.com/problems/n-queens-ii/) · Hard
<!-- practice:end -->

In [ ]:
s.finish(lookups=0, score=None, notes="")